# Chapter 1 — Self-Sufficient Notebook: Foundations Cognitive Loop Agent

This notebook converts the chapter's most important snippet into a runnable, self-sufficient Jupyter notebook.

## Why this snippet
I selected the **customer service cognitive loop** because it is the clearest concrete implementation of the chapter's core architectural idea:

- **perception**
- **reasoning**
- **planning**
- **action**
- **learning**

That loop is the conceptual backbone for the rest of the book.

## What this notebook demonstrates

- How raw user input becomes structured perception
- How intent and priority are inferred
- How an action plan is created
- How actions are executed against mocked tools
- How feedback updates preferences and learning signals

Everything here runs locally with mocked services, so no API key is needed.

In [ ]:
from __future__ import annotations

from datetime import datetime
from typing import Dict, Any, List

## Step 1 — Mock dependencies

The chapter shows simplified snippets like `analyze_sentiment`, `classify_intent`, and `billing_api.get_account(...)`.

Here we implement lightweight local versions so the architecture is fully runnable.

In [ ]:
class BillingAPI:
    def get_account(self, user_id: str) -> Dict[str, Any]:
        return {
            "user_id": user_id,
            "account_status": "active",
            "current_balance": 42.75,
            "recent_charges": [
                {"date": "2026-03-02", "amount": 19.99, "description": "Service charge"},
                {"date": "2026-03-15", "amount": 22.76, "description": "Usage adjustment"},
            ],
        }

billing_api = BillingAPI()

def analyze_sentiment(user_message: str) -> str:
    text = user_message.lower()
    negative_tokens = ["can't", "cannot", "down", "problem", "issue", "angry", "urgent", "broken"]
    if any(tok in text for tok in negative_tokens):
        return "negative"
    return "neutral"

def classify_intent(message: str) -> str:
    text = message.lower()
    if "bill" in text or "charge" in text or "invoice" in text:
        return "billing_issue"
    if "internet" in text or "connect" in text or "down" in text:
        return "technical_issue"
    return "general_support"

def get_user_history(user_id: str) -> Dict[str, Any]:
    return {
        "past_tickets": 2,
        "preferred_channel": "chat",
        "recent_sentiment_average": 0.65,
    }

def determine_priority(intent: str, sentiment: str, user_history: Dict[str, Any]) -> str:
    if intent == "technical_issue" and sentiment == "negative":
        return "urgent"
    if intent == "billing_issue":
        return "high"
    return "normal"

def calculate_success(user_feedback: Dict[str, Any]) -> float:
    return user_feedback.get("score", 0.8)

def update_user_preferences(user_id: str, success_score: float) -> None:
    print(f"[Learning] Updated preferences for {user_id} with score={success_score:.2f}")

def flag_for_model_improvement(interaction_data: Dict[str, Any]) -> None:
    print(f"[Learning] Flagged interaction for review: {interaction_data['message'][:60]}")

## Step 2 — Perception

This mirrors the book's `perceive_input(...)` function.

In [ ]:
def perceive_input(user_message: str, context: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "message": user_message,
        "timestamp": datetime.now(),
        "user_id": context.get("user_id"),
        "session_state": context.get("session"),
        "sentiment": analyze_sentiment(user_message),
    }

## Step 3 — Reasoning

This turns perception into structured understanding.

In [ ]:
def reason_about_intent(perception_data: Dict[str, Any]) -> Dict[str, Any]:
    intent = classify_intent(perception_data["message"])
    priority = determine_priority(
        intent,
        perception_data["sentiment"],
        user_history=get_user_history(perception_data["user_id"]),
    )
    return {
        "intent": intent,
        "priority": priority,
        "context": perception_data,
    }

## Step 4 — Planning

This is the book's next architectural step: convert reasoning into a sequence of actions.

In [ ]:
def create_action_plan(reasoning_result: Dict[str, Any]) -> List[str]:
    if reasoning_result["intent"] == "billing_issue":
        return [
            "fetch_account_details",
            "analyze_billing_history",
            "generate_explanation",
            "offer_resolution",
        ]
    elif reasoning_result["priority"] == "urgent":
        return [
            "escalate_to_human",
            "log_urgent_case",
        ]
    else:
        return [
            "search_help_knowledge",
            "generate_standard_reply",
        ]

## Step 5 — Action

This executes the plan against mocked tools and response generators.

In [ ]:
def generate_explanation(context: Dict[str, Any], prior_results: List[Any]) -> str:
    account = None
    for result in prior_results:
        if isinstance(result, dict) and "recent_charges" in result:
            account = result
            break

    if account:
        return (
            f"Your account is active. Your current balance is ${account['current_balance']:.2f}. "
            "I found two recent charges and can walk you through each of them."
        )
    return "I reviewed your account and can explain the recent activity."

def offer_resolution(context: Dict[str, Any], prior_results: List[Any]) -> str:
    return "I can open a billing review ticket or summarize the charges here in chat."

def execute_action(action_plan: List[str], context: Dict[str, Any]) -> List[Any]:
    results = []
    for action in action_plan:
        if action == "fetch_account_details":
            result = billing_api.get_account(context["user_id"])
        elif action == "analyze_billing_history":
            result = {"analysis": "No duplicate charges found; one usage adjustment detected."}
        elif action == "generate_explanation":
            result = generate_explanation(context, results)
        elif action == "offer_resolution":
            result = offer_resolution(context, results)
        elif action == "escalate_to_human":
            result = "Escalated to a human support specialist."
        elif action == "log_urgent_case":
            result = {"status": "urgent_case_logged"}
        elif action == "search_help_knowledge":
            result = {"article": "Restart router and verify modem lights."}
        elif action == "generate_standard_reply":
            result = "I found a support article that may help resolve this issue."
        else:
            result = f"Unknown action: {action}"

        results.append(result)
    return results

## Step 6 — Learning

This closes the loop with outcome-based improvement.

In [ ]:
def learn_from_outcome(interaction_data: Dict[str, Any], user_feedback: Dict[str, Any]) -> None:
    success_score = calculate_success(user_feedback)
    update_user_preferences(interaction_data["user_id"], success_score)
    if success_score < 0.7:
        flag_for_model_improvement(interaction_data)

## Step 7 — End-to-end cognitive loop

In [ ]:
def run_cognitive_loop(user_message: str, context: Dict[str, Any], user_feedback: Dict[str, Any]) -> Dict[str, Any]:
    perception = perceive_input(user_message, context)
    reasoning = reason_about_intent(perception)
    action_plan = create_action_plan(reasoning)
    action_results = execute_action(action_plan, perception)
    learn_from_outcome(perception, user_feedback)

    return {
        "perception": perception,
        "reasoning": reasoning,
        "plan": action_plan,
        "results": action_results,
    }

## Step 8 — Run a billing example

In [ ]:
context = {
    "user_id": "cust_1001",
    "session": {"channel": "web_chat", "conversation_id": "conv_001"}
}

user_message = "I think there's a strange charge on my bill and I can't understand it."
user_feedback = {"score": 0.86}

run_output = run_cognitive_loop(user_message, context, user_feedback)
run_output

## Step 9 — Read the output clearly

In [ ]:
print("PERCEPTION")
print("-" * 60)
for k, v in run_output["perception"].items():
    print(f"{k}: {v}")

print("\nREASONING")
print("-" * 60)
for k, v in run_output["reasoning"].items():
    if k != "context":
        print(f"{k}: {v}")

print("\nACTION PLAN")
print("-" * 60)
for step in run_output["plan"]:
    print("-", step)

print("\nACTION RESULTS")
print("-" * 60)
for item in run_output["results"]:
    print("-", item)

## Optional experiment — try a technical issue

This shows how the same architecture changes behavior based on intent and priority.

In [ ]:
tech_message = "My internet is down and I can't connect at all."
tech_feedback = {"score": 0.62}

tech_output = run_cognitive_loop(tech_message, context, tech_feedback)
tech_output["plan"], tech_output["results"]

## Why this is the chapter's most important snippet

This notebook captures the chapter's most important idea:

An agent is not just a model response.  
It is a **repeatable cognitive architecture** where:
- perception structures input,
- reasoning interprets it,
- planning chooses next steps,
- action changes the environment,
- learning improves future behavior.

That pattern is what later chapters keep expanding.

## Good extensions

- replace rule-based intent classification with an LLM
- add persistent memory across sessions
- connect real APIs for account lookup and ticket creation
- add safety checks and escalation policies
- convert this into a planning or memory-augmented agent